- mp.Process 是 Python 多进程模块（无论是原生的 multiprocessing 还是 PyTorch 的 torch.multiprocessing）中最基础、最核心的进程类。
- <font color='red'>它的作用就像是一个“工厂”，专门用来在程序中创建和启动一个全新的、独立的子进程。</font>
- 你可以把它理解为和 Python 的线程 threading.Thread 非常相似，只不过它创建的是进程而不是线程。

## 🛠️ 核心参数与用法
创建一个 Process 对象通常需要传入以下几个核心参数：
- target：目标函数。<font color='red'>也就是你希望这个新进程启动后去执行的具体任务（函数名）。</font>
- args：传递给目标函数的参数，<font color='red'>必须以元组（tuple）的形式传入（例如 (1,)）。</font>
- kwargs：传递给目标函数的关键字参数（<font color='red'>字典形式</font>）。

创建好对象后，需要调用两个关键方法：
- .start()：<font color='red'>正式触发操作系统创建新进程，并在新进程中运行 target 里的代码。</font>
- .join()：让主进程（也就是你的主程序）在这里“等待”，直到这个子进程彻底执行完毕再继续往下走。

In [ ]:
import torch.multiprocessing as mp
import time

def worker(name, delay):
    """子进程要执行的任务函数"""
    print(f"🚀 子进程 '{name}' 开始工作 (PID: {mp.current_process().pid})")
    time.sleep(delay)  # 模拟耗时操作
    print(f"✅ 子进程 '{name}' 工作结束")

if __name__ == "__main__":
    # 在涉及 PyTorch/GPU 时，强烈建议将启动方式设为 'spawn'
    mp.set_start_method('spawn')

    print("🏠 主进程开始")

    # 1. 创建两个子进程对象
    p1 = mp.Process(target=worker, args=("任务A", 2))
    p2 = mp.Process(target=worker, args=("任务B", 1))

    # 2. 启动子进程（此时 p1 和 p2 会与主进程并行运行）
    p1.start()
    p2.start()

    # 3. 等待子进程结束
    p1.join()
    p2.join()

    print("🏠 主进程结束，所有子进程已完成")

## 🧠 进阶用法：继承 Process 类
- 除了传入 target 函数，更面向对象（OOP）的做法是直接继承 mp.Process 类，并重写它的 run() 方法（<font color='red'>start() 触发后会自动调用 run()</font>）：

In [ ]:
import torch.multiprocessing as mp

class MyCustomProcess(mp.Process):
    def __init__(self, rank):
        super().__init__()
        self.rank = rank  # 自定义的实例变量

    def run(self):
        # 这里写子进程的具体逻辑
        print(f"自定义进程类启动，当前 rank 是: {self.rank}")

if __name__ == "__main__":
    mp.set_start_method('spawn')
    p = MyCustomProcess(rank=5)
    p.start()
    p.join()

## ⚠️ 两个极其重要的注意事项
1. <font color='red'>内存完全隔离（变量不共享）</font>
- 子进程是通过复制父进程的状态（或启动全新解释器）来运行的，它们拥有完全独立的内存空间。在子进程中修改任何变量，都不会影响到主进程。

- 如果需要进程间共享数据，必须使用 mp.Queue、mp.Pipe 或者你之前了解到的 tensor.share_memory_()。

In [ ]:
data = 0
def change_data():
    global data
    data = 100  # 修改的是子进程内存里的 data

p = mp.Process(target=change_data)
p.start()
p.join()
print(data)  # 输出依然是 0，主进程的 data 根本没变

2. 必须加 if __name__ == "__main__":
- 在 Windows 和 macOS（以及 Python 3.14+ 的 Linux）上，默认的进程启动方式是 spawn。<font color='red'>这意味着子进程会重新导入你的主脚本。如果不加这个保护语句，子进程在导入时会再次执行创建进程的代码，导致无限递归创建子进程，最终报错崩溃。</font>

## 总结一下：
- mp.Process 是手动控制多进程的基石。<font color='green'>虽然 PyTorch 提供了更高级的 mp.spawn 来方便地启动多卡训练，但在处理一些自定义的、非对称的后台任务时，mp.Process 依然是最灵活的工具。</font>

# spawn 启动方式

这段关于 spawn 启动方式和 if __name__ == "__main__": 保护语句的解释，触及了 Python 多进程编程中最核心、也最容易踩坑的底层机制。
为了让你彻底理解，我们可以把这个过程拆解为“克隆 vs 新生”的对比，以及“无限递归的灾难现场”来逐步分析：
1. 核心差异：克隆 (fork) vs 新生 (spawn)
在 Python 中，创建子进程主要有两种模式：
- fork（克隆模式）：这是 Linux 早期的默认方式。它就像复印机，直接把当前父进程的内存状态、代码位置、变量值原封不动地“复印”一份给子进程。子进程一出生就知道自己该从哪行代码接着往下跑，不需要重新读取脚本文件。
- spawn（新生模式）：这是 Windows、macOS 以及现代 Python（3.14+ Linux）的默认方式。它非常“干净”，但也非常“重”。子进程启动时，会开启一个全新的、空白的 Python 解释器。这个新解释器为了知道要干什么，必须重新导入（import）并执行你的主脚本文件。

2. 灾难现场：如果不加 if __name__ == "__main__": 会发生什么？
假设你在 Windows 上写了这样一段没有保护语句的代码：




In [ ]:
# 危险代码示例
import torch.multiprocessing as mp

def worker():
    print("干活中...")

# 注意：这行代码直接写在了脚本的顶层（没有被 if 保护）
p = mp.Process(target=worker)
p.start()

当主进程运行这段代码时，它会执行 p.start() 试图创建一个子进程。接下来，灾难性的无限递归就开始了：
1. 主进程 执行 p.start()。
2. 系统使用 spawn 模式，启动一个子进程A。
3. 子进程A 为了获取 worker 函数的定义，重新导入（执行）了这个脚本文件。
4. 脚本从上往下执行，又遇到了 p = mp.Process(target=worker) 和 p.start()。
5. 子进程A 开始创建孙子进程B...
6. 孙子进程B 再次导入脚本，又去创建重孙进程C...

这个过程会无限循环下去，直到你的电脑内存被耗尽，或者系统抛出 RuntimeError 或 RecursionError 报错崩溃。

3. 救星：if __name__ == "__main__": 是如何阻断递归的？
加上保护语句后，代码变成了这样：

In [ ]:
import torch.multiprocessing as mp

def worker():
    print("干活中...")

# 加上保护盾
if __name__ == "__main__":
    p = mp.Process(target=worker)
    p.start()

这里的魔法在于 Python 的 __name__ 变量：
- 当主进程直接运行脚本时：__name__ 的值是 "__main__"，<font color='green'>条件成立，p.start() 正常执行。</font>
- <font color='blue'>当子进程重新导入这个脚本时：Python 会把这个脚本当作一个普通的模块（module）来加载，此时 __name__ 的值是脚本的文件名（例如 "my_script"），而不再是 "__main__"。</font>

因此，<font color='green'>子进程在导入脚本时，if 条件判断为假，里面的 p.start() 根本不会被执行。</font>递归链条在这里被成功切断！子进程只是安静地导入了 worker 函数的定义，然后主进程就可以顺利地把这个函数交给子进程去运行了。